# Part 4 - Tabular Data (MiniBooNE Particle Identification)
Neural networks, especially CNNs, work extremely well on images. However, on tabular data, they are not as effective (see [this paper](https://proceedings.neurips.cc/paper_files/paper/2022/hash/0378c7692da36807bdec87ab043cdadc-Abstract-Datasets_and_Benchmarks.html)). Tabular data has columns of features which can have different data types like numerical or categorical values.

In this task, the goal is to achieve well performing classifier on the tabular [MiniBoone dataset](https://archive.ics.uci.edu/dataset/199/miniboone+particle+identification). You can reuse all your code from part 3.

In [5]:
import requests
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import pandas as pd
from scipy.io import arff

# Download MiniBooNE (ARFF)
download_data = [{"url": "https://www.dropbox.com/scl/fi/ht6ybxai15nzuf6jqczys/MiniBooNE.arff?rlkey=v69auyh8866mew4j02cet4b5m&st=nt0w9d74&dl=1",
                  "file": "MiniBooNE.arff"}]

root = Path("raw_miniboone"); root.mkdir(parents=True, exist_ok=True)
for d in download_data:
    p = root / d["file"]
    if not p.exists():
        r = requests.get(d["url"], timeout=60)
        r.raise_for_status()
        p.write_bytes(r.content)

# Load into DataFrame
miniboone_df = pd.DataFrame(arff.loadarff(root / "MiniBooNE.arff")[0])
miniboone_df.head()

,signal,ParticleID_0,ParticleID_1,ParticleID_2,ParticleID_3,ParticleID_4,ParticleID_5,ParticleID_6,ParticleID_7,ParticleID_8,...,ParticleID_40,ParticleID_41,ParticleID_42,ParticleID_43,ParticleID_44,ParticleID_45,ParticleID_46,ParticleID_47,ParticleID_48,ParticleID_49
0,b'True',2.59413,0.468803,20.6916,0.322648,0.009682,0.374393,0.803479,0.896592,3.59665,...,101.174,-31.3730,0.442259,5.86453,0.000000,0.090519,0.176909,0.457585,0.071769,0.245996
1,b'True',3.86388,0.645781,18.1375,0.233529,0.030733,0.361239,1.069740,0.878714,3.59243,...,186.516,45.9597,-0.478507,6.11126,0.001182,0.091800,-0.465572,0.935523,0.333613,0.230621
2,b'True',3.38584,1.197140,36.0807,0.200866,0.017341,0.260841,1.108950,0.884405,3.43159,...,129.931,-11.5608,-0.297008,8.27204,0.003854,0.141721,-0.210559,1.013450,0.255512,0.180901
3,b'True',4.28524,0.510155,674.2010,0.281923,0.009174,0.000000,0.998822,0.823390,3.16382,...,163.978,-18.4586,0.453886,2.48112,0.000000,0.180938,0.407968,4.341270,0.473081,0.258990
4,b'True',5.93662,0.832993,59.8796,0.232853,0.025066,0.233556,1.370040,0.787424,3.66546,...,229.555,42.9600,-0.975752,2.66109,0.000000,0.170836,-0.814403,4.679490,1.924990,0.253893


In [2]:
# Convert to numpy data and labels
X_unfiltered = miniboone_df.drop(columns="signal").to_numpy()
y_unfiltered = (miniboone_df["signal"] == b'True').to_numpy().astype(int)
print(f"{X_unfiltered.shape=}, {y_unfiltered.shape=}")

X_unfiltered.shape=(130064, 50), y_unfiltered.shape=(130064,)


## 4.1 Familiarize Yourself with the Data
Read the dataset description and analyze the data.
- Identify and filter out invalid samples: Locate all the rows where all features (columns) have the value -999. Remove them.
- Split the filtered data into training and testing sets. Use 20% of the data for testing and set the random_state to 42. Use the [train_test_split function from scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
- Which normalization method would you use for this dataset? Why?
- Which metrics would you use to evaluate the model?

## 4.2 Classical Machine Learning
- Try to achieve a high classification accuracy using non-neural network methods. You should aim for at least 0.937 accuracy. Discuss the results.

## 4.3 Neural Networks
- Try to get a high classification accuracy using neural networks.
- See how your neural networks perform compared to your previously used methods.